In [1]:
# === SESSION BOOTSTRAP (run first in every notebook) ======================
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, sys
PARENT = "/content/drive/MyDrive/UAV_TRUST_Research"
REPO   = f"{PARENT}/uav-trust-research"
for fname in (".gitconfig", ".git-credentials"):
    src = os.path.join(PARENT, fname)
    if os.path.exists(src):
        subprocess.run(f'cp "{src}" /root/{fname}', shell=True)
subprocess.run("git config --global credential.helper store", shell=True)
r = subprocess.run("git config --global user.name && git config --global user.email",
                   shell=True, capture_output=True, text=True)
print("git identity:", r.stdout.strip() or "MISSING - run 00_setup.ipynb first")
if os.path.isdir(REPO):
    os.chdir(REPO); sys.path.insert(0, REPO) if REPO not in sys.path else None
    print("cwd:", os.getcwd())
else:
    print("Repository not on Drive yet - run 00_setup.ipynb first.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
git identity: Md Anas Biswas
anasbiswas@gmail.com
cwd: /content/drive/MyDrive/UAV_TRUST_Research/uav-trust-research


In [2]:
!pip install xgboost shap scikit-learn matplotlib pandas numpy scipy pyarrow requests --quiet

In [3]:
# Write the memory-managed src/tsfs.py (src/data.py is already committed and correct)
import base64
open("src/tsfs.py", "w").write(base64.b64decode("IiIiVHJhbnNmZXItU3RhYmxlIEZlYXR1cmUgU2VsZWN0aW9uIChUU0ZTKS4KClNlbGVjdHMgZmVhdHVyZXMgdGhhdCBnZW5lcmFsaXplIGFjcm9zcyBhdHRhY2sgZmFtaWxpZXMgdXNpbmcgT05MWSB0aGUgc2VlbgpmYW1pbGllcywgdmlhIGFuIGludGVybmFsIGxlYXZlLW9uZS1zZWVuLWZhbWlseS1vdXQgZXN0aW1hdGUgb2YgYXR0cmlidXRpb24KcmV2ZXJzYWwuIE1lbW9yeS1tYW5hZ2VkOiBlYWNoIGludGVybmFsIG1vZGVsIGFuZCBTSEFQIGV4cGxhaW5lciBpcyByZWxlYXNlZAppbW1lZGlhdGVseSwgc2luY2UgdGhlIG5lc3RlZCBsb29wIG90aGVyd2lzZSBhY2N1bXVsYXRlcyBuYXRpdmUgbWVtb3J5LgoiIiIKaW1wb3J0IGdjCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgc2hhcAoKCmRlZiBtZWFuX3NoYXAoZXhwbGFpbmVyLCBYKToKICAgIHMgPSBucC5hc2FycmF5KGV4cGxhaW5lci5zaGFwX3ZhbHVlcyhYKSkKICAgIGlmIHMubmRpbSA9PSAzOgogICAgICAgIHMgPSBzWy4uLiwgMV0KICAgIHJldHVybiBzLm1lYW4oMCkKCgpkZWYgcmV2ZXJzYWxfdmVjdG9yKG1fcmVmZXJlbmNlLCBtX3RhcmdldCk6CiAgICAiIiJQZXItZmVhdHVyZSByZXZlcnNhbDogcHJvLWF0dGFjayBvbiByZWZlcmVuY2UsIHByby1ub3JtYWwgb24gdGFyZ2V0LiIiIgogICAgcmV0dXJuIG5wLm1heGltdW0obnAuYXNhcnJheShtX3JlZmVyZW5jZSksIDAuMCkgKiBucC5tYXhpbXVtKC1ucC5hc2FycmF5KG1fdGFyZ2V0KSwgMC4wKQoKCmRlZiBpbnRlcm5hbF9mcmFnaWxpdHkoZml0X21vZGVsLCBYLCB5LCBmYW0sIG5vcm1hbF92YWx1ZSwgc2Vlbl9mYW1pbGllcywKICAgICAgICAgICAgICAgICAgICAgICBuX2ZlYXR1cmVzLCBuX3NoYXA9MTAwMCwgc2VlZD0wKToKICAgICIiIkVzdGltYXRlIHBlci1mZWF0dXJlIHRyYW5zZmVyLWZyYWdpbGl0eSB1c2luZyBvbmx5IHRoZSBzZWVuIGZhbWlsaWVzLgoKICAgIGZpdF9tb2RlbChYX3RyLCB5X3RyLCBzZWVkKSByZXR1cm5zIGEgZml0dGVkIHRyZWUgY2xhc3NpZmllciB3aXRoIHByZWRpY3RfcHJvYmEuCiAgICBFYWNoIGZvbGQncyBtb2RlbCBhbmQgZXhwbGFpbmVyIGFyZSBkZWxldGVkIGJlZm9yZSB0aGUgbmV4dCB0byBib3VuZCBtZW1vcnkuCiAgICAiIiIKICAgIGZhbSA9IG5wLmFzYXJyYXkoZmFtKQogICAgYWNjID0gbnAuemVyb3Mobl9mZWF0dXJlcykKICAgIHJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhzZWVkKQogICAgZm9yIGksIGYgaW4gZW51bWVyYXRlKHNlZW5fZmFtaWxpZXMpOgogICAgICAgIG90aGVycyA9IFtnIGZvciBnIGluIHNlZW5fZmFtaWxpZXMgaWYgZyAhPSBmXQogICAgICAgIHRyID0gKGZhbSA9PSBub3JtYWxfdmFsdWUpIHwgbnAuaXNpbihmYW0sIG90aGVycykKICAgICAgICBjbGYgPSBmaXRfbW9kZWwoWFt0cl0sIHlbdHJdLCBzZWVkICsgaSkKICAgICAgICBleHBsID0gc2hhcC5UcmVlRXhwbGFpbmVyKGNsZikKCiAgICAgICAgZGVmIHNhbXAobWFzayk6CiAgICAgICAgICAgIGlkeCA9IG5wLndoZXJlKG1hc2spWzBdCiAgICAgICAgICAgIGlmIGxlbihpZHgpID4gbl9zaGFwOgogICAgICAgICAgICAgICAgaWR4ID0gcm5nLmNob2ljZShpZHgsIG5fc2hhcCwgcmVwbGFjZT1GYWxzZSkKICAgICAgICAgICAgcmV0dXJuIFhbaWR4XQoKICAgICAgICBtX3JlZiA9IG1lYW5fc2hhcChleHBsLCBzYW1wKG5wLmlzaW4oZmFtLCBvdGhlcnMpKSkKICAgICAgICBtX2YgPSBtZWFuX3NoYXAoZXhwbCwgc2FtcChmYW0gPT0gZikpCiAgICAgICAgYWNjICs9IHJldmVyc2FsX3ZlY3RvcihtX3JlZiwgbV9mKQogICAgICAgIGRlbCBjbGYsIGV4cGwKICAgICAgICBnYy5jb2xsZWN0KCkKICAgIHJldHVybiBhY2MgLyBtYXgobGVuKHNlZW5fZmFtaWxpZXMpLCAxKQoKCmRlZiBzZWxlY3Rfc3RhYmxlX2ZlYXR1cmVzKGZyYWdpbGl0eSwgZHJvcF9mcmFjdGlvbj0wLjIpOgogICAgIiIiSW5kaWNlcyBvZiB0cmFuc2Zlci1zdGFibGUgZmVhdHVyZXMgKGRyb3AgdGhlIG1vc3QtZnJhZ2lsZSBmcmFjdGlvbikuIiIiCiAgICBmcmFnaWxpdHkgPSBucC5hc2FycmF5KGZyYWdpbGl0eSkKICAgIHAgPSBsZW4oZnJhZ2lsaXR5KQogICAgbl9kcm9wID0gaW50KHJvdW5kKGRyb3BfZnJhY3Rpb24gKiBwKSkKICAgIGlmIG5fZHJvcCA8PSAwOgogICAgICAgIHJldHVybiBucC5hcmFuZ2UocCkKICAgIGRyb3AgPSBzZXQobnAuYXJnc29ydCgtZnJhZ2lsaXR5KVs6bl9kcm9wXS50b2xpc3QoKSkKICAgIHJldHVybiBucC5hcnJheShbaiBmb3IgaiBpbiByYW5nZShwKSBpZiBqIG5vdCBpbiBkcm9wXSkK").decode())
print("src/tsfs.py updated (memory-managed internal loop)")

src/tsfs.py updated (memory-managed internal loop)


In [4]:
# Imports (reload tsfs so the memory-managed version is active)
import importlib, gc, src.tsfs
importlib.reload(src.tsfs)
import numpy as np, pandas as pd, requests, glob, zipfile
import matplotlib.pyplot as plt
import xgboost as xgb, shap
from scipy.stats import spearmanr
from sklearn.metrics import balanced_accuracy_score
from src.data import load_csvs, detect_schema, prepare_splits
from src.tsfs import mean_shap, reversal_vector, internal_fragility, select_stable_features
from src.trust import top_label_ece, conformal_qhat
print("imports ready")

imports ready


In [5]:
# Config: full data, ip.proto dropped, single seed (nested LOFO), smaller SHAP sample
DATASETS = [
  {"name": "UAVIDS-2025", "kind": "zenodo", "record": "15336998",
   "data_dir": "data/uavids2025", "label_col": "label", "normal_value": "Normal Traffic",
   "include_families": None, "subsample_n": None,
   "drops": ["unnamed","flowid","srcaddr","dstaddr","srcport","dstport","index","timestamp"]},
  {"name": "UAV-NIDD", "kind": "file",
   "file": "data/uav_nidd/UAV-NDD CSV/UAV-Case1-Label.csv", "parquet": "data/uav_nidd/case1.parquet",
   "data_dir": "data/uav_nidd", "label_col": "Label", "normal_value": "Normal",
   "include_families": ["DDoS","UDP Flooding","MITM","Jamming","BruteForce","De-authentication"],
   "subsample_n": 300000,
   "drops": ["unnamed","index","ip.src","ip.dst","ip.proto","srcport","dstport","udp.srcport","udp.dstport",
             "frame.time","frame.number","time_epoch","time_relative","time_delta","bssid","mactime",
             "vendor_oui","wlan_radio.timestamp","wlan_radio.start_tsf","radiotap.timestamp","wlan.seq"]},
]
CFG = {"seed": 0, "n_shap": 1000, "alpha": 0.10, "nbins": 15,
       "drop_fractions": [0.20],
       "normal_fracs": {"train":0.60,"cal":0.20,"test_seen":0.10,"test_shift":0.10},
       "family_fracs": {"train":0.60,"cal":0.20,"test_seen":0.20},
       "xgb": {"n_estimators":300,"max_depth":6,"learning_rate":0.1,"subsample":0.9,
               "colsample_bytree":0.9,"tree_method":"hist"},
       "fig_dir":"figures", "report_dir":"reports"}
for d in [CFG["fig_dir"], CFG["report_dir"]]: os.makedirs(d, exist_ok=True)
print("configured")

configured


In [6]:
# Loader and helpers
def load_dataset(spec):
    dd = spec["data_dir"]; os.makedirs(dd, exist_ok=True)
    if spec["kind"] == "zenodo":
        if not glob.glob(dd+"/**/*.csv", recursive=True):
            meta = requests.get(f"https://zenodo.org/api/records/{spec['record']}", timeout=60).json()
            for f in meta.get("files", []):
                n,u = f["key"], f["links"]["self"]
                if n.lower().endswith((".csv",".zip",".gz")):
                    open(os.path.join(dd,n),"wb").write(requests.get(u,timeout=1200).content)
            for z in glob.glob(dd+"/*.zip"): zipfile.ZipFile(z).extractall(dd)
        df = load_csvs(dd); lc,nv,fams = detect_schema(df, spec["label_col"], spec["normal_value"])
    else:
        pq = spec.get("parquet")
        if pq and os.path.exists(pq): df = pd.read_parquet(pq)
        else:
            df = pd.read_csv(spec["file"], low_memory=False, encoding="latin-1")
            if pq:
                try: df.to_parquet(pq)
                except Exception as e: print("parquet skip:", e)
        lc,nv = spec["label_col"], spec["normal_value"]; fams = [v for v in df[lc].unique() if v!=nv]
    if spec.get("subsample_n") and len(df) > spec["subsample_n"]:
        frac = spec["subsample_n"]/len(df)
        df = df.groupby(lc, group_keys=False).sample(frac=frac, random_state=42).reset_index(drop=True)
    if spec.get("include_families"):
        df = df[df[lc].isin([nv]+list(spec["include_families"]))].reset_index(drop=True)
        fams = list(spec["include_families"])
    return df, lc, nv, fams

def make_fitter():
    return lambda Xtr, ytr, sd: xgb.XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                                                  random_state=sd, **CFG["xgb"]).fit(Xtr, ytr)

def samp(X, mask, seed=0):
    idx = np.where(mask)[0]
    if len(idx) > CFG["n_shap"]:
        idx = np.random.default_rng(seed).choice(idx, CFG["n_shap"], replace=False)
    return X[idx]

def coverage_detail(p, y, qhat):
    p2 = np.column_stack([1-np.asarray(p), np.asarray(p)]); sets = p2 >= (1-qhat); y = np.asarray(y)
    inset = sets[np.arange(len(y)), y]
    return float(inset.mean()), float(np.mean([inset[y==k].mean() for k in np.unique(y)])), float(sets.sum(1).mean())

def evaluate(clf, X_cal, y_cal, X_shift, y_shift):
    p_cal = clf.predict_proba(X_cal)[:,1]; p_shift = clf.predict_proba(X_shift)[:,1]
    qhat = conformal_qhat(p_cal, y_cal, alpha=CFG["alpha"])
    _, bcov, _ = coverage_detail(p_shift, y_shift, qhat)
    ba = balanced_accuracy_score(y_shift, (p_shift>=0.5).astype(int))
    ece = top_label_ece(p_shift, y_shift, CFG["nbins"])
    return ba, bcov, ece
print("helpers ready")

helpers ready


In [ ]:
# TSFS experiment with aggressive memory release (float32 matrices, del + gc each step)
fit = make_fitter()
val_rows, res_rows = [], []
for spec in DATASETS:
    df, lc, nv, fams = load_dataset(spec)
    for F in fams:
        S = prepare_splits(df, lc, nv, F, spec["drops"], CFG["normal_fracs"], CFG["family_fracs"], CFG["seed"])
        for kk in ["X_train","X_cal","X_seen","X_shift"]:
            S[kk] = S[kk].astype(np.float32)                      # halve matrix memory
        seen = [g for g in fams if g != F]; nfeat = S["X_train"].shape[1]
        internal = internal_fragility(fit, S["X_train"], S["y_train"], S["fam_train"],
                                      nv, seen, nfeat, n_shap=CFG["n_shap"], seed=CFG["seed"])
        base = fit(S["X_train"], S["y_train"], CFG["seed"])
        ex = shap.TreeExplainer(base)
        oracle = reversal_vector(mean_shap(ex, samp(S["X_seen"], np.isin(S["fam_seen"], seen))),
                                 mean_shap(ex, samp(S["X_shift"], S["fam_shift"]==F)))
        rho = spearmanr(internal, oracle).correlation
        k = max(3, int(0.1*nfeat)); ti=set(np.argsort(-internal)[:k]); to=set(np.argsort(-oracle)[:k])
        jac = len(ti & to)/len(ti | to)
        val_rows.append({"dataset": spec["name"], "held_out": str(F), "n_features": nfeat,
                         "rho_internal_oracle": round(float(rho),3), "topk_jaccard": round(jac,3)})
        ba_b, cov_b, ece_b = evaluate(base, S["X_cal"], S["y_cal"], S["X_shift"], S["y_shift"])
        del ex; gc.collect()
        for dfrac in CFG["drop_fractions"]:
            keep = select_stable_features(internal, dfrac)
            Xtr_k = S["X_train"][:,keep]
            tclf = fit(Xtr_k, S["y_train"], CFG["seed"])
            ba_t, cov_t, ece_t = evaluate(tclf, S["X_cal"][:,keep], S["y_cal"], S["X_shift"][:,keep], S["y_shift"])
            res_rows.append({"dataset": spec["name"], "held_out": str(F), "drop_fraction": dfrac, "n_kept": len(keep),
                             "cov_baseline": round(cov_b,4), "cov_tsfs": round(cov_t,4), "cov_delta": round(cov_t-cov_b,4),
                             "balacc_baseline": round(ba_b,4), "balacc_tsfs": round(ba_t,4),
                             "ece_baseline": round(ece_b,4), "ece_tsfs": round(ece_t,4)})
            del tclf, Xtr_k, keep; gc.collect()
        print(spec["name"], str(F), "done  (rho_int_oracle=%.2f)"%rho)
        del S, base, internal; gc.collect()
    del df; gc.collect()
val = pd.DataFrame(val_rows); res = pd.DataFrame(res_rows)
val.to_csv(os.path.join(CFG["report_dir"],"09_tsfs_selection_validity.csv"), index=False)
res.to_csv(os.path.join(CFG["report_dir"],"09_tsfs_results.csv"), index=False)
print("done")

UAVIDS-2025 Sybil Attack done  (rho_int_oracle=0.10)
UAVIDS-2025 Blackhole Attack done  (rho_int_oracle=0.24)
UAVIDS-2025 Wormhole Attack done  (rho_int_oracle=0.54)
UAVIDS-2025 Flooding Attack done  (rho_int_oracle=0.48)
UAV-NIDD DDoS done  (rho_int_oracle=0.76)
UAV-NIDD UDP Flooding done  (rho_int_oracle=0.78)
UAV-NIDD MITM done  (rho_int_oracle=0.67)
UAV-NIDD Jamming done  (rho_int_oracle=0.73)
UAV-NIDD BruteForce done  (rho_int_oracle=0.83)


In [ ]:
# VALIDATION: can fragile features be identified without the unseen family?
print("Internal (seen-only) vs oracle (unseen) fragility agreement:")
print(val.to_string(index=False))
print("\nmean Spearman:", round(val["rho_internal_oracle"].mean(),3),
      " mean top-k Jaccard:", round(val["topk_jaccard"].mean(),3))

In [ ]:
# RESULT: does TSFS improve trust on the unseen family? (balanced conformal coverage)
piv = res.pivot_table(index=["dataset","held_out"], columns="drop_fraction", values="cov_delta").round(4)
piv.columns = ["cov_delta@%.2f"%c for c in piv.columns]
print("Change in shift balanced-coverage, TSFS minus baseline (positive = TSFS better):")
print(piv.to_string())
print("\nMean coverage delta by drop fraction:")
print(res.groupby("drop_fraction")[["cov_delta"]].mean().round(4).to_string())
print("\nMean balanced-accuracy: baseline %.4f  tsfs %.4f" % (res["balacc_baseline"].mean(), res["balacc_tsfs"].mean()))

In [ ]:
# FIGURE: internal-vs-oracle agreement and baseline-vs-TSFS coverage
best = res[res["drop_fraction"]==0.20]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
ax[0].bar(range(len(val)), val["rho_internal_oracle"], color="#00695c")
ax[0].set_xticks(range(len(val))); ax[0].set_xticklabels([r["held_out"][:9] for _,r in val.iterrows()], rotation=35, ha="right", fontsize=7)
ax[0].axhline(0, color="gray", lw=.6); ax[0].set_ylabel("Spearman (internal vs oracle)")
ax[0].set_title("Fragility is identifiable from seen families alone")
x = np.arange(len(best)); w = 0.4
ax[1].bar(x-w/2, best["cov_baseline"], w, label="baseline (all features)", color="#c62828")
ax[1].bar(x+w/2, best["cov_tsfs"], w, label="TSFS (stable features)", color="#2e7d32")
ax[1].axhline(1-CFG["alpha"], ls="--", color="gray", lw=1)
ax[1].set_xticks(x); ax[1].set_xticklabels([r["held_out"][:9] for _,r in best.iterrows()], rotation=35, ha="right", fontsize=7)
ax[1].set_ylabel("shift balanced coverage"); ax[1].set_title("TSFS vs baseline under attack-family shift (drop=0.20)")
ax[1].legend(fontsize=8)
fig.tight_layout(); fig.savefig(os.path.join(CFG["fig_dir"],"09_tsfs.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Commit
!git add src/ reports/ figures/ notebooks/
!git commit -m "09 TSFS (final): memory-managed run on full data, ip.proto dropped, seen-only selection validated vs oracle"
!git push origin main